# DeepSeek-R1 reinforcement learning: GRPO from first principles

This notebook implements the core Group Relative Policy Optimization (GRPO) loop used in DeepSeek-R1 on a tiny autoregressive policy. It uses CPU PyTorch and no RL training framework.

**Anchor paper:** DeepSeek-AI et al., [DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning](https://arxiv.org/abs/2501.12948), arXiv:2501.12948v2. Related version of record: [Nature 645, 633-638 (2025)](https://www.nature.com/articles/s41586-025-09422-z). The local PDF is `papers/deepseek-r1-arxiv-2501.12948v2.pdf`.

## Learning objectives

1. Express autoregressive generation as a policy acting over tokens.
2. Generate multiple completions per prompt and score them with verifiable rewards.
3. Construct group-relative advantages without a learned critic.
4. Implement the clipped policy-ratio objective and reference-policy KL penalty.
5. Track reward, exact-answer accuracy, formatting, entropy, KL, clipping, and zero-variance groups.
6. Explain what this controlled experiment does and does not reproduce from DeepSeek-R1.

## 1. Keep the paper's pipelines separate

### DeepSeek-R1-Zero

R1-Zero starts from DeepSeek-V3-Base and applies reasoning-oriented RL without preliminary SFT. Its automatically checked rewards incentivize correct answers and the requested output structure. The experiment shows that a strong base model can discover useful long-form reasoning behavior under outcome rewards, but it also develops readability and language-mixing problems.

### DeepSeek-R1

R1 is a multi-stage system: cold-start SFT, reasoning RL, rejection sampling plus a larger SFT stage, and another RL stage covering broader helpfulness and safety objectives. Therefore, R1's final performance cannot be attributed to GRPO alone.

This notebook most closely resembles a very small **R1-Zero-style algorithm experiment**: start from a randomly initialized policy, sample groups, apply rule-based rewards, and update with GRPO. A random tiny policy is not comparable to a pretrained LLM.

## 2. Translate language-model generation into RL

| RL term | Autoregressive LLM interpretation |
|---|---|
| Environment/context | Prompt $q$ and any verifier state |
| State $s_t$ | Prompt plus response prefix $o_{<t}$ |
| Action $a_t$ | Next response token $o_t$ |
| Policy $\pi_\theta$ | Decoder-only LM token distribution |
| Trajectory $o$ | Complete sampled response |
| Reward $r(q,o)$ | Verifier, format check, reward model, or combination |

The policy factorizes a response as

$$\pi_\theta(o\mid q)=\prod_{t=1}^{|o|}\pi_\theta(o_t\mid q,o_{<t}).$$

Unlike SFT, the response tokens are sampled from the policy. The reward evaluates the sampled trajectory; it does not provide a ground-truth token at every position. Policy gradients assign credit to sampled tokens that made high-reward trajectories more likely.

In [1]:
from __future__ import annotations

import copy
import random

import torch
from torch import nn
from torch.distributions import Categorical
from torch.nn import functional as F

SEED = 11
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

print(f"PyTorch {torch.__version__}; device=cpu")

PyTorch 2.13.0+cpu; device=cpu


## 3. A verifiable toy environment

Each prompt contains two integers from 0 through 4. A completion contains exactly two actions:

```text
answer-token  <eos>
```

The verifier assigns `1.0` for the exact sum and `0.1` for the correct EOS format. There are no target response tokens in the RL update. The verifier only scores what the policy sampled.

This environment deliberately makes correctness objective and cheap. DeepSeek-R1 similarly concentrates reasoning RL on domains such as mathematics and code where answers can be checked, but its prompts, completions, base policy, and infrastructure are vastly more complex.

In [2]:
MAX_INPUT = 4
MAX_ANSWER = 8
EOS = 9
INVALID = 10
NUM_ACTIONS = 11
START = 11  # input-only marker; the policy never emits it
RESPONSE_LENGTH = 2
ACTION_NAMES = [str(value) for value in range(MAX_ANSWER + 1)] + ["<eos>", "<invalid>"]

prompts = torch.tensor(
    [(left, right) for left in range(MAX_INPUT + 1) for right in range(MAX_INPUT + 1)],
    dtype=torch.long,
)

def score_completions(prompt_batch: torch.Tensor, actions: torch.Tensor):
    target = prompt_batch[:, 0] + prompt_batch[:, 1]
    correct = actions[:, 0].eq(target)
    correct_format = actions[:, 1].eq(EOS)
    rewards = correct.float() + 0.1 * correct_format.float()
    return rewards, correct, correct_format

def render_completion(actions: torch.Tensor) -> str:
    return " ".join(ACTION_NAMES[action] for action in actions.tolist())

print(f"Prompts: {len(prompts)}; action vocabulary: {ACTION_NAMES}")

Prompts: 25; action vocabulary: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '<eos>', '<invalid>']


## 4. Tiny autoregressive policy

The teaching policy uses a GRU cell rather than repeating the Transformer implementation from the first notebook. It is still autoregressive: the second action is conditioned on the prompt and first sampled action. GRPO only needs token probabilities and gradients, so the optimizer is unchanged when this policy is replaced by a decoder-only Transformer LLM.

During rollout, actions are sampled. During the GRPO update, the sampled actions are teacher-forced through the policy so their current log-probabilities can be recomputed.

In [3]:
class TinyAutoregressivePolicy(nn.Module):
    def __init__(self, hidden_size: int = 48):
        super().__init__()
        self.number_embedding = nn.Embedding(MAX_INPUT + 1, hidden_size)
        self.action_embedding = nn.Embedding(NUM_ACTIONS + 1, hidden_size)
        self.initial_state = nn.Sequential(
            nn.Linear(2 * hidden_size, hidden_size),
            nn.Tanh(),
        )
        self.recurrent_cell = nn.GRUCell(hidden_size, hidden_size)
        self.policy_head = nn.Linear(hidden_size, NUM_ACTIONS)

    def prompt_state(self, prompt_batch: torch.Tensor) -> torch.Tensor:
        left = self.number_embedding(prompt_batch[:, 0])
        right = self.number_embedding(prompt_batch[:, 1])
        return self.initial_state(torch.cat([left, right], dim=-1))

    def step(self, prompt_batch, previous_action, hidden_state=None):
        if hidden_state is None:
            hidden_state = self.prompt_state(prompt_batch)
        hidden_state = self.recurrent_cell(
            self.action_embedding(previous_action), hidden_state
        )
        return self.policy_head(hidden_state), hidden_state

    def forward(self, prompt_batch: torch.Tensor, actions: torch.Tensor):
        hidden_state = None
        previous_action = torch.full(
            (len(prompt_batch),), START, dtype=torch.long, device=prompt_batch.device
        )
        logits = []
        for position in range(RESPONSE_LENGTH):
            position_logits, hidden_state = self.step(
                prompt_batch, previous_action, hidden_state
            )
            logits.append(position_logits)
            previous_action = actions[:, position]
        return torch.stack(logits, dim=1)


def action_log_probs(policy, prompt_batch, actions):
    logits = policy(prompt_batch, actions)
    return F.log_softmax(logits, dim=-1).gather(
        dim=-1, index=actions.unsqueeze(-1)
    ).squeeze(-1)


@torch.no_grad()
def sample_groups(policy, unique_prompts, group_size, temperature=1.0):
    prompt_count = len(unique_prompts)
    expanded_prompts = (
        unique_prompts[:, None, :]
        .expand(prompt_count, group_size, 2)
        .reshape(-1, 2)
    )
    previous_action = torch.full((len(expanded_prompts),), START, dtype=torch.long)
    hidden_state = None
    actions, log_probs, entropies = [], [], []
    for _ in range(RESPONSE_LENGTH):
        logits, hidden_state = policy.step(expanded_prompts, previous_action, hidden_state)
        distribution = Categorical(logits=logits / temperature)
        previous_action = distribution.sample()
        actions.append(previous_action)
        log_probs.append(distribution.log_prob(previous_action))
        entropies.append(distribution.entropy())
    return (
        expanded_prompts,
        torch.stack(actions, dim=1),
        torch.stack(log_probs, dim=1),
        torch.stack(entropies, dim=1),
    )


@torch.no_grad()
def greedy_evaluate(policy, prompt_batch):
    previous_action = torch.full((len(prompt_batch),), START, dtype=torch.long)
    hidden_state = None
    actions = []
    for _ in range(RESPONSE_LENGTH):
        logits, hidden_state = policy.step(prompt_batch, previous_action, hidden_state)
        previous_action = logits.argmax(dim=-1)
        actions.append(previous_action)
    actions = torch.stack(actions, dim=1)
    rewards, correct, correct_format = score_completions(prompt_batch, actions)
    return {
        "reward": rewards.mean().item(),
        "accuracy": correct.float().mean().item(),
        "format": correct_format.float().mean().item(),
        "actions": actions,
    }


policy = TinyAutoregressivePolicy()
parameter_count = sum(parameter.numel() for parameter in policy.parameters())
print(f"Policy parameters: {parameter_count:,}")
print("Initial greedy metrics:", {k: v for k, v in greedy_evaluate(policy, prompts).items() if k != "actions"})

Policy parameters: 20,123
Initial greedy metrics: {'reward': 0.03999999910593033, 'accuracy': 0.03999999910593033, 'format': 0.0}


## 5. Group-relative advantages

Before discussing optimization, keep the unit of each quantity clear:

- A **token** is one action.
- A **trajectory** is one complete sampled response containing many token actions.
- A **group** is several trajectories sampled for the same prompt.
- The verifier returns one **reward per trajectory**, not one reward per token.
- GRPO converts each trajectory reward into one **advantage per trajectory**.

For one prompt, sample $G$ trajectories and obtain rewards $r_1,\ldots,r_G$. GRPO uses the other trajectories for that prompt as a baseline:

$$A_i = \frac{r_i-\mu_r}{\sigma_r+\epsilon_A}, \qquad \mu_r=\frac{1}{G}\sum_jr_j.$$

For example, suppose four trajectories are sampled for `2 + 3`:

| trajectory | sampled response | reward | advantage | meaning |
|---:|---|---:|---:|---|
| 1 | `5 <eos>` | $1.1$ | approximately $+1$ | better than this group's average |
| 2 | `5 <eos>` | $1.1$ | approximately $+1$ | better than this group's average |
| 3 | `4 <eos>` | $0.1$ | approximately $-1$ | worse than this group's average |
| 4 | `6 <eos>` | $0.1$ | approximately $-1$ | worse than this group's average |

Here $\mu_r=0.6$ and $\sigma_r=0.5$. The advantage is therefore a relative label attached to the **whole row**. It does not yet say how to update the model, and it does not claim that any particular token caused the outcome.

A completion is positive only relative to other samples for the same prompt. This avoids training a separate value critic, but introduces important behavior:

- If every completion has the same reward, every advantage is zero and that prompt supplies no policy-gradient signal.
- Very easy and very hard prompts can therefore become uninformative.
- Larger groups estimate relative outcomes more reliably but require more rollout generation.
- Reward scale within each group is normalized away; reward ordering remains.

In [4]:
def group_relative_advantages(rewards: torch.Tensor, prompt_count: int, group_size: int):
    grouped = rewards.view(prompt_count, group_size)
    means = grouped.mean(dim=1, keepdim=True)
    standard_deviations = grouped.std(dim=1, keepdim=True, unbiased=False)
    normalized = (grouped - means) / (standard_deviations + 1e-4)
    normalized = torch.where(standard_deviations > 1e-6, normalized, 0.0)
    return normalized.reshape(-1), standard_deviations.squeeze(1)

demo_prompt = torch.tensor([[2, 2]])
demo_group_size = 8
demo_prompts, demo_actions, _, _ = sample_groups(policy, demo_prompt, demo_group_size)
demo_rewards, _, _ = score_completions(demo_prompts, demo_actions)
demo_advantages, demo_std = group_relative_advantages(
    demo_rewards, len(demo_prompt), demo_group_size
)
print("Samples for prompt 2 + 2:")
for actions, reward, advantage in zip(demo_actions, demo_rewards, demo_advantages):
    print(f"  {render_completion(actions):13s} reward={reward.item():.1f} advantage={advantage.item():+.3f}")
print(f"group reward standard deviation={demo_std.item():.3f}")

Samples for prompt 2 + 2:
  2 8           reward=0.0 advantage=-0.378
  2 2           reward=0.0 advantage=-0.378
  6 0           reward=0.0 advantage=-0.378
  <eos> 8       reward=0.0 advantage=-0.378
  3 6           reward=0.0 advantage=-0.378
  0 8           reward=0.0 advantage=-0.378
  2 6           reward=0.0 advantage=-0.378
  4 6           reward=1.0 advantage=+2.645
group reward standard deviation=0.331


### 5.1 How one trajectory advantage drives an update

An advantage becomes useful only when it weights log-probabilities produced by the trainable model. Ignore clipping and KL for a moment. A basic trajectory-level policy-gradient objective is

$$J_i(\theta)=A_i\frac{1}{T_i}\sum_{t=1}^{T_i}\log\pi_\theta(o_{i,t}\mid q,o_{i,<t}), \qquad \mathcal{L}_i=-J_i.$$

Read this from the inside out:

1. Replay trajectory $i$ through the current model and retrieve the log-probability of each token that was actually sampled.
2. Average those token log-probabilities to obtain a score for that sampled trajectory.
3. Multiply the score by the trajectory advantage $A_i$. The same $A_i$ is broadcast across all tokens in this trajectory.
4. Negate the objective because the optimizer minimizes a loss.
5. Backpropagate the scalar batch loss through every token log-probability into the shared transformer parameters.

The sign of $A_i$ determines the update direction:

- If $A_i>0$, minimizing $-A_i\log\pi_\theta(o_{i,t})$ increases the probability of each sampled token in that trajectory.
- If $A_i<0$, the sign reverses, so minimizing the loss decreases the probability of each sampled token in that trajectory.
- If $A_i=0$, the trajectory contributes no reward-driven gradient.

For `5 <eos>` with positive advantage, both the sampled `5` and the sampled `<eos>` are reinforced under their respective prefixes. For `4 <eos>` with negative advantage, both sampled actions are suppressed. The model is not told that `5` alone was the important decision. This is the limitation of outcome-level credit assignment.

The transformer parameters are shared, so this does not behave like independently editing a table entry for each token. Gradients from all tokens and trajectories are added together. They can reinforce or oppose one another, and the parameter update changes probabilities in many other contexts through generalization.

#### One complete optimization cycle

1. Freeze a snapshot $\pi_{\mathrm{old}}$ and use it to sample several trajectories per prompt. Sampling itself has no backpropagation.
2. Store the exact sampled token IDs and their old log-probabilities.
3. Score each complete trajectory and compute one group-relative advantage $A_i$ for it. Advantages are treated as fixed numbers during the update.
4. Replay the stored tokens through the current $\pi_\theta$ with teacher forcing to obtain current per-token log-probabilities.
5. Broadcast each trajectory's advantage over its tokens, construct their objective terms, and reduce all terms to one scalar loss.
6. Call `loss.backward()`. The chain rule carries each weighted token signal through the output head and transformer layers.
7. Call `optimizer.step()`. Positive-advantage sampled decisions tend to become more likely; negative-advantage sampled decisions tend to become less likely.
8. After the configured optimization epochs, collect fresh on-policy trajectories and repeat.

The $1/T_i$ above makes each trajectory an equal-sized unit regardless of length. Another implementation may average over all valid tokens in the batch instead, in which case longer trajectories contribute more terms. This notebook uses fixed two-token trajectories, so the two reductions are equivalent up to a batch-wide constant.

The code cell below isolates the sign mechanism. Its two advantages sum to zero, so the displayed scalar loss starts at zero. Nevertheless, the gradients do not vanish: the good and bad trajectories contain different sampled actions, so they differentiate different log-probabilities.


In [5]:
# Two trajectories, two token positions, and a three-token toy vocabulary.
toy_logits = torch.zeros(2, 2, 3, requires_grad=True)
sampled_token_ids = torch.tensor([[0, 2], [1, 2]])
trajectory_advantages = torch.tensor([+1.0, -1.0])

toy_log_probs = F.log_softmax(toy_logits, dim=-1)
sampled_log_probs = toy_log_probs.gather(-1, sampled_token_ids[..., None]).squeeze(-1)
token_advantages = trajectory_advantages[:, None]  # Broadcast one value over each trajectory.
toy_loss = -(token_advantages * sampled_log_probs).mean()
toy_loss.backward()

with torch.no_grad():
    learning_rate = 0.5
    updated_logits = toy_logits - learning_rate * toy_logits.grad
    probabilities_before = F.softmax(toy_logits, dim=-1)
    probabilities_after = F.softmax(updated_logits, dim=-1)
    sampled_before = probabilities_before.gather(-1, sampled_token_ids[..., None]).squeeze(-1)
    sampled_after = probabilities_after.gather(-1, sampled_token_ids[..., None]).squeeze(-1)

print(f"scalar loss before the update: {toy_loss.item():+.4f}")
for index, label in enumerate(["good trajectory, A=+1", "bad trajectory, A=-1"]):
    print(label)
    print(f"  sampled-token probabilities before: {sampled_before[index].tolist()}")
    print(f"  sampled-token probabilities after:  {sampled_after[index].tolist()}")

scalar loss before the update: -0.0000
good trajectory, A=+1
  sampled-token probabilities before: [0.3333333432674408, 0.3333333432674408]
  sampled-token probabilities after:  [0.36166447401046753, 0.36166447401046753]
bad trajectory, A=-1
  sampled-token probabilities before: [0.3333333432674408, 0.3333333432674408]
  sampled-token probabilities after:  [0.30615708231925964, 0.30615708231925964]


## 6. GRPO clipped objective and KL control

### 6.1 The three policies have different jobs

Keep these models separate mentally:

- $\pi_{\mathrm{old}}$ is the **behavior policy** that generated this rollout batch. It stays fixed while the batch is optimized.
- $\pi_\theta$ is the **current trainable policy**. Its probabilities change after optimizer steps.
- $\pi_{\mathrm{ref}}$ is the **frozen reference policy**, usually the model at the start of RL. It is a longer-term anchor.

Clipping compares the current policy with the policy that collected the batch. The KL term compares the current policy with the starting model. They control different kinds of movement.

### 6.2 First ask: did this sampled token become more likely?

For the exact token $o_{i,t}$ that appeared in the rollout, under the same prompt and prefix, compute

$$\rho_{i,t}(\theta)=\exp\left(\log\pi_\theta(o_{i,t}\mid q,o_{i,<t})-\log\pi_{\mathrm{old}}(o_{i,t}\mid q,o_{i,<t})\right)=\frac{\pi_\theta(o_{i,t}\mid q,o_{i,<t})}{\pi_{\mathrm{old}}(o_{i,t}\mid q,o_{i,<t})}.$$

Suppose the old policy assigned the sampled token probability $0.20$, while the current policy assigns it $0.30$. Then $\rho=0.30/0.20=1.50$. The current model has made that decision 50% more likely. A ratio of $1$ means no change; a ratio below $1$ means the token became less likely. This ratio concerns the sampled token, not the whole vocabulary.

On the first optimization step, $\pi_\theta=\pi_{\mathrm{old}}$, so every ratio is numerically $1$. This does **not** mean the gradient is zero. The old log-probability is stored and detached, while the current log-probability remains differentiable:

$$\nabla_\theta\rho_{i,t}=\rho_{i,t}\nabla_\theta\log\pi_\theta(o_{i,t}\mid q,o_{i,<t}).$$

Therefore, at $\rho=1$, the gradient of $\rho A_i$ is exactly $A_i\nabla_\theta\log\pi_\theta$. The advantage already pushes good sampled actions up and bad sampled actions down. After one optimizer step, the ratios move away from $1$; clipping then limits how much more the same stored batch can move the policy during later optimization steps.

### 6.3 The advantage supplies the direction

GRPO gives completion $i$ a group-relative advantage $A_i$. If $A_i>0$, the completion did better than its peers and its sampled actions should become more likely. If $A_i<0$, they should become less likely. Without clipping, the token contribution would be $\rho_{i,t}A_i$.

The problem is that repeatedly optimizing one batch could make its sampled tokens extremely more or less likely, even though the advantage is only an estimate from that batch. With $\epsilon=0.20$, GRPO therefore constructs a second candidate using a ratio restricted to $[0.8,1.2]$:

$$L^{\mathrm{clip}}_{i,t}=\min\left(\rho_{i,t}A_i,\operatorname{clip}(\rho_{i,t},0.8,1.2)A_i\right).$$

The optimizer maximizes this surrogate, so `min` chooses the more conservative, lower-valued candidate. Consider both signs of the advantage:

| case | $A_i$ | old $p$ | current $p$ | $\rho$ | raw $\rho A$ | clipped candidate | selected |
|---|---:|---:|---:|---:|---:|---:|---:|
| good completion | $+1.5$ | $0.20$ | $0.30$ | $1.50$ | $2.25$ | $1.20(1.5)=1.80$ | $1.80$ |
| bad completion | $-1.0$ | $0.20$ | $0.10$ | $0.50$ | $-0.50$ | $0.80(-1)=-0.80$ | $-0.80$ |

In the first row, making a good action 50% more likely receives no more credit than making it 20% more likely. In the second row, making a bad action 50% less likely receives no more credit than making it 20% less likely: if the ratio falls further, the selected value remains $-0.80$. Clipping limits extra reward for moving too far in the desired direction. It does **not** hide movement in the wrong direction; such movement still worsens the objective.

### 6.4 KL adds a long-term anchor

Clipping only keeps this update close to $\pi_{\mathrm{old}}$. Because the next batch uses a newer behavior policy, many individually small updates could still drift far from the original model. The frozen $\pi_{\mathrm{ref}}$ addresses that accumulation. For the sampled token, define

$$d_{i,t}=\log\pi_{\mathrm{ref}}(o_{i,t})-\log\pi_\theta(o_{i,t}),$$
$$\widehat D_{\mathrm{KL},i,t}=\exp(d_{i,t})-d_{i,t}-1.$$

Continue the good-completion example with reference probability $0.25$ and current probability $0.30$. Then $d=\log(0.25/0.30)\approx-0.1823$ and the sampled KL estimate is about $0.0157$. With $\beta=0.04$, the token objective is $1.80-0.04(0.0157)\approx1.7994$. The reward signal wins here, but policy drift has a small cost.

The quantity being maximized is

$$J_{i,t}=L^{\mathrm{clip}}_{i,t}-\beta\widehat D_{\mathrm{KL},i,t}.$$

PyTorch optimizers minimize, so the implementation returns the negative mean, $\mathcal{L}=-\operatorname{mean}_{i,t}J_{i,t}$. A larger useful objective therefore produces a smaller loss.

### 6.5 Where token-level credit stops

The advantage $A_i$ evaluates the **whole completion**, but it is broadcast to every sampled token in that completion. Each token has its own probability ratio and KL cost, yet all receive the same outcome direction. A successful trajectory therefore reinforces all of its sampled actions; GRPO does not identify which reasoning step caused success. That is outcome-level credit assignment, not process-level credit assignment.

In [6]:
def clipped_token_example(
    label: str,
    old_probability: float,
    current_probability: float,
    advantage: float,
    reference_probability: float = 0.25,
    epsilon: float = 0.20,
    beta: float = 0.04,
):
    old_p = torch.tensor(old_probability)
    current_p = torch.tensor(current_probability)
    reference_p = torch.tensor(reference_probability)
    ratio = current_p / old_p
    raw_surrogate = ratio * advantage
    clipped_surrogate_candidate = ratio.clamp(1 - epsilon, 1 + epsilon) * advantage
    selected_surrogate = torch.minimum(raw_surrogate, clipped_surrogate_candidate)

    reference_minus_policy = torch.log(reference_p) - torch.log(current_p)
    sampled_kl = torch.exp(reference_minus_policy) - reference_minus_policy - 1
    objective = selected_surrogate - beta * sampled_kl
    loss = -objective
    return {
        "case": label,
        "ratio": ratio.item(),
        "raw": raw_surrogate.item(),
        "clipped_candidate": clipped_surrogate_candidate.item(),
        "selected": selected_surrogate.item(),
        "sampled_kl": sampled_kl.item(),
        "objective": objective.item(),
        "loss": loss.item(),
    }


walkthrough = [
    clipped_token_example("good completion", 0.20, 0.30, +1.5),
    clipped_token_example("bad completion", 0.20, 0.10, -1.0),
]

for row in walkthrough:
    print(row["case"])
    print(
        f"  ratio={row['ratio']:.2f}, raw={row['raw']:.2f}, "
        f"clipped candidate={row['clipped_candidate']:.2f}, selected={row['selected']:.2f}"
    )
    print(
        f"  sampled KL={row['sampled_kl']:.4f}, "
        f"objective={row['objective']:.4f}, minimized loss={row['loss']:.4f}"
    )

good completion
  ratio=1.50, raw=2.25, clipped candidate=1.80, selected=1.80
  sampled KL=0.0157, objective=1.7994, minimized loss=-1.7994
bad completion
  ratio=0.50, raw=-0.50, clipped candidate=-0.80, selected=-0.80
  sampled KL=0.5837, objective=-0.8233, minimized loss=0.8233


In [7]:
def grpo_loss(
    policy,
    prompt_batch,
    actions,
    old_log_probs,
    reference_log_probs,
    advantages,
    clip_epsilon,
    kl_coefficient,
):
    current_log_probs = action_log_probs(policy, prompt_batch, actions)
    ratios = torch.exp(current_log_probs - old_log_probs)
    token_advantages = advantages[:, None]
    unclipped = ratios * token_advantages
    clipped = ratios.clamp(1 - clip_epsilon, 1 + clip_epsilon) * token_advantages
    clipped_surrogate = torch.minimum(unclipped, clipped)

    reference_minus_policy = reference_log_probs - current_log_probs
    sampled_kl = (
        torch.exp(reference_minus_policy) - reference_minus_policy - 1
    )
    objective = clipped_surrogate - kl_coefficient * sampled_kl
    loss = -objective.mean()
    diagnostics = {
        "loss": loss.item(),
        "kl": sampled_kl.mean().item(),
        "clip_fraction": (ratios.sub(1).abs() > clip_epsilon).float().mean().item(),
        "mean_ratio": ratios.mean().item(),
    }
    return loss, diagnostics


## 7. Complete on-policy training loop

Each update has two phases:

1. **Rollout:** freeze the current policy as the behavior policy for this batch, sample $G$ completions per prompt, store their old log-probabilities, and compute rewards and advantages.
2. **Optimization:** reuse those trajectories for several epochs. Recompute current log-probabilities, apply clipping and KL, backpropagate, clip gradients, and update the policy.

The reference policy is a frozen copy of the initial policy. We store old log-probabilities rather than a second old-policy model because the rollout batch is small and synchronous. Large systems must additionally manage generation/training workers, stale rollouts, sequence packing, distributed parameter updates, and reference inference.

In [8]:
reference_policy = copy.deepcopy(policy).eval()
for parameter in reference_policy.parameters():
    parameter.requires_grad_(False)

optimizer = torch.optim.AdamW(policy.parameters(), lr=1.5e-2, weight_decay=1e-3)
GROUP_SIZE = 16
CLIP_EPSILON = 0.2
KL_COEFFICIENT = 0.02
OPTIMIZATION_EPOCHS = 4
UPDATES = 151
history = []

for update in range(UPDATES):
    # Rollout: these tensors define pi_old for the current batch.
    rollout_prompts, actions, old_log_probs, entropies = sample_groups(
        policy, prompts, GROUP_SIZE
    )
    rewards, correct, correct_format = score_completions(rollout_prompts, actions)
    advantages, reward_stds = group_relative_advantages(
        rewards, len(prompts), GROUP_SIZE
    )
    old_log_probs = old_log_probs.detach()
    with torch.no_grad():
        reference_log_probs = action_log_probs(
            reference_policy, rollout_prompts, actions
        )

    # Optimization: multiple gradient epochs over the fixed rollout batch.
    for _ in range(OPTIMIZATION_EPOCHS):
        loss, diagnostics = grpo_loss(
            policy,
            rollout_prompts,
            actions,
            old_log_probs,
            reference_log_probs,
            advantages,
            CLIP_EPSILON,
            KL_COEFFICIENT,
        )
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        gradient_norm = nn.utils.clip_grad_norm_(policy.parameters(), max_norm=1.0)
        optimizer.step()

    greedy = greedy_evaluate(policy, prompts)
    record = {
        "update": update,
        "rollout_reward": rewards.mean().item(),
        "greedy_accuracy": greedy["accuracy"],
        "greedy_format": greedy["format"],
        "entropy": entropies.mean().item(),
        "zero_std_groups": (reward_stds < 1e-6).float().mean().item(),
        "gradient_norm": float(gradient_norm),
        **diagnostics,
    }
    history.append(record)
    if update % 10 == 0:
        print(
            f"u={update:3d} reward={record['rollout_reward']:.3f} "
            f"accuracy={record['greedy_accuracy']:.2f} format={record['greedy_format']:.2f} "
            f"KL={record['kl']:.3f} entropy={record['entropy']:.3f} "
            f"zero-groups={record['zero_std_groups']:.2f}"
        )

final_metrics = greedy_evaluate(policy, prompts)
assert final_metrics["accuracy"] == 1.0
assert final_metrics["format"] == 1.0
print("Final greedy accuracy and format checks: 100%")

u=  0 reward=0.102 accuracy=0.40 format=0.36 KL=0.077 entropy=2.386 zero-groups=0.08


u= 10 reward=1.008 accuracy=0.96 format=0.96 KL=2.335 entropy=0.463 zero-groups=0.40


u= 20 reward=1.051 accuracy=0.96 format=0.96 KL=1.976 entropy=0.201 zero-groups=0.60


u= 30 reward=1.079 accuracy=1.00 format=0.96 KL=3.116 entropy=0.135 zero-groups=0.60


u= 40 reward=1.084 accuracy=1.00 format=0.96 KL=2.777 entropy=0.125 zero-groups=0.68


...

u=110 reward=1.089 accuracy=1.00 format=1.00 KL=2.046 entropy=0.080 zero-groups=0.68


u=120 reward=1.087 accuracy=1.00 format=1.00 KL=2.763 entropy=0.052 zero-groups=0.72


u=130 reward=1.100 accuracy=1.00 format=1.00 KL=1.617 entropy=0.046 zero-groups=0.92


u=140 reward=1.095 accuracy=1.00 format=1.00 KL=1.860 entropy=0.043 zero-groups=0.84


u=150 reward=1.094 accuracy=1.00 format=1.00 KL=1.743 entropy=0.051 zero-groups=0.88
Final greedy accuracy and format checks: 100%


In [9]:
final_actions = final_metrics["actions"]
for prompt, actions in zip(prompts, final_actions):
    left, right = prompt.tolist()
    print(f"{left} + {right} -> {render_completion(actions)}")

0 + 0 -> 0 <eos>
0 + 1 -> 1 <eos>
0 + 2 -> 2 <eos>
0 + 3 -> 3 <eos>
0 + 4 -> 4 <eos>
1 + 0 -> 1 <eos>
1 + 1 -> 2 <eos>
1 + 2 -> 3 <eos>
1 + 3 -> 4 <eos>
1 + 4 -> 5 <eos>
2 + 0 -> 2 <eos>
2 + 1 -> 3 <eos>
2 + 2 -> 4 <eos>
2 + 3 -> 5 <eos>
2 + 4 -> 6 <eos>
3 + 0 -> 3 <eos>
3 + 1 -> 4 <eos>
3 + 2 -> 5 <eos>
3 + 3 -> 6 <eos>
3 + 4 -> 7 <eos>
4 + 0 -> 4 <eos>
4 + 1 -> 5 <eos>
4 + 2 -> 6 <eos>
4 + 3 -> 7 <eos>
4 + 4 -> 8 <eos>


## 8. Map the notebook to DeepSeek-R1

| Notebook | DeepSeek-R1 analogue | Important difference |
|---|---|---|
| Two-integer prompt | Math, code, and STEM prompt | Real prompts require long semantic reasoning |
| Two-token GRU policy | DeepSeek-V3-Base / R1 policy | Real policy is a very large decoder-only MoE Transformer |
| 16 samples per prompt | Grouped response rollouts | Real completions may contain thousands of tokens |
| Exact sum reward | Rule-based accuracy verifier | Real verifiers cover answers, programs, and task-specific checks |
| EOS reward | Structural format reward | R1 uses explicit reasoning/final-answer structure and other pipeline controls |
| Group reward normalization | GRPO group-relative baseline | Same concept, radically different scale and sampling distribution |
| Frozen initial policy | Reference model | Real reference inference is a major systems cost |
| Stored rollout log-probs | Old behavior policy | Distributed rollouts introduce policy staleness |
| Greedy exact accuracy | pass@1-style evaluation | Reasoning evaluation also studies sampling, robustness, style, and contamination |

The notebook demonstrates that the estimator can optimize a sparse sequence reward. It does not show emergent reasoning, self-reflection, inference-time scaling, or generalization. The policy memorizes a finite table of arithmetic prompts.

## 9. Compute and storage implications

GRPO removes the learned critic/value model used in conventional PPO, but RL training remains substantially more expensive than SFT.

### Models and stored tensors

- **Trainable policy:** weights, gradients, optimizer states, and training activations.
- **Reference policy:** frozen weights plus forward activations needed only transiently.
- **Old policy:** may be a rollout snapshot; this notebook stores old token log-probabilities instead.
- **Rollout batch:** prompt/response tokens, masks, old log-probabilities, reference log-probabilities, rewards, and advantages.
- **Generation KV cache:** scales with concurrent sequences and generated context length.

For $B$ unique prompts, group size $G$, and response length $R$, rollout generation produces approximately $BGR$ response-token decisions. Autoregressive generation is sequential in $R$. Reusing the rollout batch for $K$ optimization epochs adds roughly $K$ policy forward/backward passes over the prompt-response sequences, plus reference-policy scoring.

Removing a critic saves critic parameters, optimizer state, inference, and value activations. It does **not** remove grouped sampling, reference scoring, policy training state, or rollout storage. At LLM scale, rollout throughput, long and variable response lengths, distributed synchronization, and stale-policy control are often as important as the loss equation.

## 10. Critical questions and exercises

1. Set `GROUP_SIZE = 2`, `4`, `8`, and `32`. Compare reward variance, zero-variance groups, and updates needed for convergence.
2. Remove the format reward. Does the policy still learn EOS, and what does that say about underspecified rewards?
3. Remove the KL term, then increase it substantially. Compare exploration, convergence, and policy drift.
4. Set `OPTIMIZATION_EPOCHS` high enough to create a large clip fraction. Explain why stale rollouts eventually become unsafe to reuse.
5. Corrupt the verifier for a subset of prompts. Observe reward hacking: the policy optimizes the implemented reward, not the intended task.
6. Add prompts whose answers are outside the action vocabulary. Show why impossible groups yield no useful relative signal.
7. Create a cold-start policy with SFT on a small subset, then run the same RL loop. Compare format stability and sample efficiency with random-start RL.
8. Replace sequence-level advantages with token-specific process rewards. Identify the new supervision and failure modes required.
9. Replace the GRU with the causal Transformer from notebook 01 without changing the GRPO code.
10. Re-read the paper's unsuccessful attempts and classify each issue as reward design, search/optimization, model capacity, or infrastructure.